# Analyzing Daily Covid19 New Cases for US Counties

## Loading and preprocessing data

In [1]:
import pandas as pd

# Load the dataset
covid_us_data = pd.read_csv('time_series_covid19_confirmed_US.csv')

# Display basic information about the dataset
print("Columns in the dataset:")
print(covid_us_data.columns)

# Check for missing values
print("\nMissing values in the dataset:")
print(covid_us_data.isnull().sum())

# Display the first few rows
print("\nFirst five rows of the dataset:")
print(covid_us_data.head())

# Show the date columns for the time series data
date_columns = covid_us_data.columns[11:] 
print("\nDate columns in the dataset:")
print(date_columns)

# Display the range of dates in the dataset
print("\nDate range in the dataset:")
print(f"Start date: {date_columns[0]}")
print(f"End date: {date_columns[-1]}")

# Check for counties with missing data over the date range
missing_data = covid_us_data[date_columns].isnull().sum()
print("\nNumber of missing entries per date:")
print(missing_data)

Columns in the dataset:
Index(['UID', 'iso2', 'iso3', 'code3', 'FIPS', 'Admin2', 'Province_State',
       'Country_Region', 'Lat', 'Long_',
       ...
       '2/28/23', '3/1/23', '3/2/23', '3/3/23', '3/4/23', '3/5/23', '3/6/23',
       '3/7/23', '3/8/23', '3/9/23'],
      dtype='object', length=1154)

Missing values in the dataset:
UID        0
iso2       0
iso3       0
code3      0
FIPS      10
          ..
3/5/23     0
3/6/23     0
3/7/23     0
3/8/23     0
3/9/23     0
Length: 1154, dtype: int64

First five rows of the dataset:
        UID iso2 iso3  code3    FIPS   Admin2 Province_State Country_Region  \
0  84001001   US  USA    840  1001.0  Autauga        Alabama             US   
1  84001003   US  USA    840  1003.0  Baldwin        Alabama             US   
2  84001005   US  USA    840  1005.0  Barbour        Alabama             US   
3  84001007   US  USA    840  1007.0     Bibb        Alabama             US   
4  84001009   US  USA    840  1009.0   Blount        Alabama        

## Loading population data 

Loding population data for counties to normalize the case numbers based on each counties' population each year

In [4]:
population_data = pd.read_excel('coest2023pop.xlsx', skiprows=4)

# Clean the data: rename the columns for better understanding
population_data.columns = ['Geographic Area', '2020 Estimate Base', '2020 Population', '2021 Population', '2022 Population', '2023 Population']

# Drop any rows where the geographic area is NaN (if they exist)
population_data = population_data.dropna(subset=['Geographic Area'])

# Clean up the 'Geographic Area' column by stripping unnecessary characters
population_data['Geographic Area'] = population_data['Geographic Area'].str.replace(r'^\.', '', regex=True)

# Show the cleaned data
print(population_data.head())

           Geographic Area  2020 Estimate Base  2020 Population  \
0  Autauga County, Alabama             58809.0          58915.0   
1  Baldwin County, Alabama            231768.0         233227.0   
2  Barbour County, Alabama             25229.0          24969.0   
3     Bibb County, Alabama             22301.0          22188.0   
4   Blount County, Alabama             59130.0          59107.0   

   2021 Population  2022 Population  2023 Population  
0          59203.0          59726.0          60342.0  
1         239439.0         246531.0         253507.0  
2          24533.0          24700.0          24585.0  
3          22359.0          21986.0          21868.0  
4          59079.0          59516.0          59816.0  


Doing some preprocessing to make it possible to merge the population data with covid data for counties

In [21]:
population_data[['County', 'State']]=population_data['Geographic Area'].str.split(', ', expand=True)


In [69]:
import re

def clean_county_names(df, column_name='county'):
    words_to_remove = ['County', 'Borough', 'Census Area', 'City and Borough', 'Parish', 'city']
    
    # Create a regular expression pattern
    pattern = '|'.join(map(re.escape, words_to_remove))
    
    # Remove the words from the end of the county names
    df[column_name] = df[column_name].str.replace(f' {pattern}$', '', regex=True).str.strip()
    
    return df

population_data_cleaned = clean_county_names(population_data, 'County')

In [129]:
merged_data = covid_us_data.merge(population_data_cleaned[['County', 'State', '2020 Population', '2021 Population', '2022 Population', '2023 Population']],
                              left_on=['Admin2', 'Province_State'], right_on=['County', 'State'], how='left')

In [130]:
# Check for duplicates in the merged dataset
duplicates = merged_data[merged_data.duplicated(subset=['Admin2', 'Province_State'], keep=False)]
print(duplicates)

Empty DataFrame
Columns: [UID, iso2, iso3, code3, FIPS, Admin2, Province_State, Country_Region, Lat, Long_, Combined_Key, 1/22/20, 1/23/20, 1/24/20, 1/25/20, 1/26/20, 1/27/20, 1/28/20, 1/29/20, 1/30/20, 1/31/20, 2/1/20, 2/2/20, 2/3/20, 2/4/20, 2/5/20, 2/6/20, 2/7/20, 2/8/20, 2/9/20, 2/10/20, 2/11/20, 2/12/20, 2/13/20, 2/14/20, 2/15/20, 2/16/20, 2/17/20, 2/18/20, 2/19/20, 2/20/20, 2/21/20, 2/22/20, 2/23/20, 2/24/20, 2/25/20, 2/26/20, 2/27/20, 2/28/20, 2/29/20, 3/1/20, 3/2/20, 3/3/20, 3/4/20, 3/5/20, 3/6/20, 3/7/20, 3/8/20, 3/9/20, 3/10/20, 3/11/20, 3/12/20, 3/13/20, 3/14/20, 3/15/20, 3/16/20, 3/17/20, 3/18/20, 3/19/20, 3/20/20, 3/21/20, 3/22/20, 3/23/20, 3/24/20, 3/25/20, 3/26/20, 3/27/20, 3/28/20, 3/29/20, 3/30/20, 3/31/20, 4/1/20, 4/2/20, 4/3/20, 4/4/20, 4/5/20, 4/6/20, 4/7/20, 4/8/20, 4/9/20, 4/10/20, 4/11/20, 4/12/20, 4/13/20, 4/14/20, 4/15/20, 4/16/20, 4/17/20, 4/18/20, 4/19/20, ...]
Index: []

[0 rows x 1160 columns]


In [127]:
print(covid_us_data[covid_us_data['Admin2']=='Roanoke City'])
print(population_data_cleaned[population_data_cleaned['County']=='Roanoke'])


           UID iso2 iso3  code3     FIPS        Admin2 Province_State  \
3118  84051770   US  USA    840  51770.0  Roanoke City       Virginia   

     Country_Region       Lat      Long_  ... 2/28/23  3/1/23  3/2/23  3/3/23  \
3118             US  37.27791 -79.961898  ...   28317   28317   28317   28317   

      3/4/23  3/5/23  3/6/23  3/7/23  3/8/23  3/9/23  
3118   28317   28317   28317   28360   28360   28360  

[1 rows x 1154 columns]
               Geographic Area  2020 Estimate Base  2020 Population  \
2898  Roanoke County, Virginia             96931.0          96923.0   
2947    Roanoke city, Virginia            100014.0          99891.0   

      2021 Population  2022 Population  2023 Population   County     State  
2898          96671.0          96756.0          97026.0  Roanoke  Virginia  
2947          98700.0          97657.0          97171.0  Roanoke  Virginia  


In [128]:
population_data_cleaned.loc[population_data_cleaned['Geographic Area']=='Roanoke city, Virginia', 'County'] = 'Roanoke City'

print(population_data_cleaned[population_data_cleaned['Geographic Area']=='Roanoke city, Virginia'])


             Geographic Area  2020 Estimate Base  2020 Population  \
2947  Roanoke city, Virginia            100014.0          99891.0   

      2021 Population  2022 Population  2023 Population        County  \
2947          98700.0          97657.0          97171.0  Roanoke City   

         State  
2947  Virginia  


In [86]:
print(covid_us_data['FIPS'].isnull().sum())

10


In [87]:
# Find the extra rows in covid_us_data not present in population_data
covid_counties = covid_us_data[['Admin2', 'Province_State']].drop_duplicates()
pop_counties = population_data_cleaned[['County']].drop_duplicates()

# Check which counties are missing in population data
missing_in_pop = covid_counties[~covid_counties['Admin2'].isin(pop_counties['County'])]

print(missing_in_pop)


                                   Admin2 Province_State
52                              Out of AL        Alabama
64                             Unassigned        Alabama
71                              Anchorage         Alaska
74    Bristol Bay plus Lake and Peninsula         Alaska
91                              Out of AK         Alaska
...                                   ...            ...
3235                           Unassigned  West Virginia
3287                            Out of WI      Wisconsin
3306                           Unassigned      Wisconsin
3331                            Out of WY        Wyoming
3339                           Unassigned        Wyoming

[212 rows x 2 columns]


Separating different population for years to normalize the case numbers based on each corresponding year

In [131]:
covid_2020 = covid_us_data.loc[:, '1/22/20':'12/31/20']
covid_2021 = covid_us_data.loc[:, '1/1/21':'12/31/21']
covid_2022 = covid_us_data.loc[:, '1/1/22':'12/31/22']
covid_2023 = covid_us_data.loc[:, '1/1/23':'3/9/23']  

# Normalize for each year using the corresponding population data
# Normalize 2020 data by 2020 population
covid_2020_normalized = covid_2020.div(merged_data['2020 Population'].values, axis=0) * 1e6

# Normalize 2021 data by 2021 population
covid_2021_normalized = covid_2021.div(merged_data['2021 Population'].values, axis=0) * 1e6

# Normalize 2022 data by 2022 population
covid_2022_normalized = covid_2022.div(merged_data['2022 Population'].values, axis=0) * 1e6

# Normalize 2023 data by 2023 population
covid_2023_normalized = covid_2023.div(merged_data['2023 Population'].values, axis=0) * 1e6

In [132]:
# Combine the normalized data back together
normalized_covid_data = pd.concat([covid_2020_normalized, covid_2021_normalized, covid_2022_normalized, covid_2023_normalized], axis=1)

In [133]:
# Add back the relevant columns for context
normalized_covid_data.insert(0, 'FIPS', covid_us_data['FIPS'])
normalized_covid_data.insert(1, 'Admin2', covid_us_data['Admin2'])
normalized_covid_data.insert(2, 'State', covid_us_data['Province_State'])
normalized_covid_data.insert(3, 'Lat', covid_us_data['Lat'])
normalized_covid_data.insert(4, 'Long', covid_us_data['Long_'])

# Result: normalized_covid_data contains the daily cases normalized by county population for each year.
print(normalized_covid_data.head())

     FIPS   Admin2    State        Lat       Long  1/22/20  1/23/20  1/24/20  \
0  1001.0  Autauga  Alabama  32.539527 -86.644082      0.0      0.0      0.0   
1  1003.0  Baldwin  Alabama  30.727750 -87.722071      0.0      0.0      0.0   
2  1005.0  Barbour  Alabama  31.868263 -85.387129      0.0      0.0      0.0   
3  1007.0     Bibb  Alabama  32.996421 -87.125115      0.0      0.0      0.0   
4  1009.0   Blount  Alabama  33.982109 -86.567906      0.0      0.0      0.0   

   1/25/20  1/26/20  ...        2/28/23         3/1/23         3/2/23  \
0      0.0      0.0  ...  327002.750986  327450.200524  327450.200524   
1      0.0      0.0  ...  274710.363027  275207.390723  275207.390723   
2      0.0      0.0  ...  303070.978239  304006.508033  304006.508033   
3      0.0      0.0  ...  368895.189318  369809.767697  369809.767697   
4      0.0      0.0  ...  311221.077972  312174.000267  312174.000267   

          3/3/23         3/4/23         3/5/23         3/6/23         3/7/23  \


In [134]:
normalized_covid_data.to_pickle('normalized_covid_data.pkl')